# SiC Wafer Dicing Simulation — End-to-End Demo

**Pipeline**: ABAQUS FEM → GP Surrogate → Bayesian Optimization → TMCMC Inference

This notebook demonstrates the full workflow for optimizing SiC blade dicing parameters
using physics-based simulation and machine learning.

| Stage | Tool | Output |
|-------|------|--------|
| 1. FEM | ABAQUS/Explicit | Chipping fraction, stress field |
| 2. Surrogate | Gaussian Process | Response surface |
| 3. BO | Expected Improvement | Optimal parameters |
| 4. Inference | TMCMC | Posterior distribution |

**Material**: 4H-SiC  
**Parameters**: Cut depth [10–70 µm], Blade width [15–50 µm]

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Material Properties

In [ ]:
from data.materials.material_properties import SiC, Si, GaN

print("=== 4H-SiC ===")
for k, v in SiC.items():
    print(f"  {k:20s}: {v}")

print("\n=== Fracture energy G_c ===")
for mat in [Si, SiC, GaN]:
    Gc = mat['K_Ic']**2 / mat['E']
    print(f"  {mat['name']:12s}: G_c = {Gc:.4f} J/m²")

## 2. FEM Parametric Study

Run the ABAQUS parametric study (15 jobs: 5 depths × 3 blade widths).

```bash
# Create run directory
mkdir -p runs/parametric && cd runs/parametric

# Create config
echo '{"study": true, "material": "SiC"}' > run_config.json

# Submit all 15 jobs
abaqus cae noGUI=../../fem/dicing_blade_2d.py

# Extract results (after jobs complete)
abaqus python ../../fem/extract_results.py -- \
    --manifest jobs_4HSiC.json --odb-dir .
```

**Demo mode**: load pre-generated synthetic data

In [ ]:
# ── Synthetic FEM data (replace with real parametric_summary.csv) ─────────────
np.random.seed(42)
cut_depths  = np.array([20, 30, 40, 50, 60], dtype=float)
blade_widths = np.array([20, 30, 40], dtype=float)
D, BW = np.meshgrid(cut_depths, blade_widths)

# Physics-informed synthetic response
# Chipping increases with depth, decreases with blade width
chip = 0.002 * D.ravel() - 0.0008 * BW.ravel() + \
       0.00003 * D.ravel()**2 + np.random.randn(15) * 0.003
chip = np.clip(chip, 0.0, 0.3)

# Stress (GPa) — higher depth and narrower blade → higher stress
stress = 5.0 + 0.15 * D.ravel() - 0.05 * BW.ravel() + \
         np.random.randn(15) * 0.5

df = pd.DataFrame({
    'cut_depth_um': D.ravel(),
    'blade_W_um':   BW.ravel(),
    'deletion_fraction':        chip,
    'max_principal_stress_Pa':  stress * 1e9,
})
print(df.to_string(index=False))

In [ ]:
# Visualize raw FEM data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, label, cmap in zip(
    axes,
    ['deletion_fraction', 'max_principal_stress_Pa'],
    ['Chipping Fraction', 'Max Stress [GPa]'],
    ['Reds', 'Blues']):

    z = df[col].values
    if 'stress' in col:
        z = z / 1e9
    sc = ax.scatter(df['cut_depth_um'], df['blade_W_um'],
                    c=z, s=200, cmap=cmap, edgecolors='k')
    plt.colorbar(sc, ax=ax, label=label)
    ax.set_xlabel('Cut Depth [µm]')
    ax.set_ylabel('Blade Width [µm]')
    ax.set_title(label)

plt.tight_layout()
plt.show()

## 3. Gaussian Process Surrogate

In [ ]:
from ml.surrogate_gp import DicingGPSurrogate, FEATURE_COLS, TARGET_COLS

X = df[FEATURE_COLS].values.astype(float)
Y = df[TARGET_COLS].values.astype(float)
Y[:, 1] /= 1e9   # Pa → GPa

model = DicingGPSurrogate()
model.fit(X, Y)
print("GP fitted successfully")

In [ ]:
# Response surface
d_grid  = np.linspace(10, 70, 80)
bw_grid = np.linspace(15, 50, 80)
DD, BWW = np.meshgrid(d_grid, bw_grid)
X_grid  = np.column_stack([DD.ravel(), BWW.ravel()])

mu, sigma = model.predict(X_grid, return_std=True)

fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

titles = [
    (mu[:, 0], 'GP Mean: Chipping Fraction', 'viridis'),
    (sigma[:, 0], 'GP Std: Chipping Fraction', 'plasma'),
    (mu[:, 1], 'GP Mean: Max Stress [GPa]', 'hot'),
]
for i, (data, title, cmap) in enumerate(titles):
    ax = fig.add_subplot(gs[i])
    im = ax.contourf(DD, BWW, data.reshape(DD.shape), levels=25, cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.scatter(X[:, 0], X[:, 1], c='w', s=60, edgecolors='k', zorder=5)
    ax.set_xlabel('Cut Depth [µm]')
    ax.set_ylabel('Blade Width [µm]')
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 4. Bayesian Optimization

Minimize chipping fraction subject to stress < 0.95 × σ_fracture

In [ ]:
from optimization.bayesian_opt import (
    constrained_ei, maximize_ei, compute_pareto_front, SIGMA_FRACTURE
)

y_best = Y[:, 0].min()
result = maximize_ei(model, y_best, constrained=True)

print("=== Bayesian Optimization Result ===")
for k, v in result.items():
    print(f"  {k:35s}: {v}")

In [ ]:
# Expected Improvement landscape
ei_vals = constrained_ei(X_grid, model, y_best)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
im = ax.contourf(DD, BWW, ei_vals.reshape(DD.shape), levels=25, cmap='Greens')
plt.colorbar(im, ax=ax, label='Constrained EI')
ax.scatter(X[:, 0], X[:, 1], c='r', s=60, zorder=5, label='FEM data')
ax.scatter(result['cut_depth_um'], result['blade_W_um'],
           c='gold', s=200, marker='*', zorder=6, label='Next experiment')
ax.set_xlabel('Cut Depth [µm]')
ax.set_ylabel('Blade Width [µm]')
ax.set_title('Constrained Expected Improvement')
ax.legend()

# Pareto front (fast grid)
ax = axes[1]
mu_grid, _ = model.predict(X_grid)
ax.scatter(mu_grid[:, 0], mu_grid[:, 1],
           c=DD.ravel(), cmap='coolwarm', s=5, alpha=0.4)
ax.axvline(y_best, color='red', ls='--', label=f'Current best chip={y_best:.4f}')
ax.axhline(SIGMA_FRACTURE, color='orange', ls='--',
           label=f'Fracture limit {SIGMA_FRACTURE:.1f} GPa')
ax.set_xlabel('Chipping Fraction')
ax.set_ylabel('Max Stress [GPa]')
ax.set_title('Objective Space')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 5. TMCMC Inference

Bayesian inference: given observed chipping, what cutting parameters explain it?

In [ ]:
from optimization.tmcmc_dicing import tmcmc, log_likelihood, log_prior, PARAM_BOUNDS

observed_chip = 0.025   # e.g., from experimental measurement
print(f"Observed chipping fraction: {observed_chip}")

ll_fn = lambda theta: log_likelihood(theta, model, observed_chip, sigma_obs=0.005)
result = tmcmc(ll_fn, log_prior, n_samples=500, n_steps=15)  # quick demo

samples = result['samples']
weights = result['weights']

mean_p = np.average(samples, weights=weights, axis=0)
print(f"\nPosterior mean:")
print(f"  Cut depth: {mean_p[0]:.2f} µm")
print(f"  Blade W:   {mean_p[1]:.2f} µm")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sc = ax.scatter(samples[:, 0], samples[:, 1],
                c=weights, cmap='plasma', s=15, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Posterior weight')
ax.scatter(*mean_p, c='lime', s=200, marker='*', zorder=5, label='Posterior mean')
ax.set_xlabel('Cut Depth [µm]')
ax.set_ylabel('Blade Width [µm]')
ax.set_title(f'TMCMC Posterior | observed chip={observed_chip}')
ax.legend()

ax = axes[1]
ax.hist(samples[:, 0], bins=30, weights=weights, alpha=0.6, density=True,
        label='Cut depth [µm]')
ax2 = ax.twinx()
ax2.hist(samples[:, 1], bins=30, weights=weights, alpha=0.5, density=True,
         color='orange', label='Blade width [µm]')
ax.set_xlabel('Parameter value [µm]')
ax.set_title('Posterior Marginals')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 6. Summary

| Component | Method | Key result |
|-----------|--------|------------|
| FEM | ABAQUS/Explicit, CPE4R, MaxPS damage | Chipping + stress field |
| Surrogate | Anisotropic RBF-GP, LOO-CV | R² > 0.95 (synthetic) |
| Optimization | Constrained EI with GP | Optimal depth/kerf in seconds |
| Inference | TMCMC (Ching & Chen 2007) | Full posterior, uncertainty |

**Next steps**:
1. Replace synthetic data with real ABAQUS output from `parametric_summary.csv`
2. Train FNO on stress *fields* (`ml/surrogate_fno.py`)
3. Extend to KABRA thermal model (`fem/kabra_thermal_2d.py`)
4. Active learning loop: `optimization/bayesian_opt.py --active`